# Mini Proyecto: Reconocimiento de Dígitos Escritos a Mano
### Cloud, IA y Data Science — Ciclo 1

---

En este proyecto vamos a construir un sistema capaz de **leer dígitos escritos a mano** y reconocer qué número representan. Este es exactamente el tipo de problema que hay detrás de la lectura automática de códigos postales, cheques bancarios o formularios escaneados.

A diferencia de proyectos anteriores donde los datos eran medidas numéricas simples, aquí cada muestra es una **imagen**. Eso nos va a permitir ver cosas que antes no podíamos: los propios datos tienen una forma visual que podemos mostrar en pantalla.

El proyecto sigue el mismo flujo de trabajo estándar:

1. **Definición del problema** — ¿Qué queremos predecir?
2. **Carga y exploración** — ¿Qué pinta tienen los datos de imagen?
3. **Preprocesamiento** — ¿Cómo preparamos imágenes para un modelo?
4. **Entrenamiento** — Usaremos un algoritmo distinto al de proyectos anteriores
5. **Evaluación** — Con 10 clases posibles, la evaluación se complica
6. **Reflexión** — ¿Qué tiene de especial trabajar con imágenes?

---
## Fase 1 — Definición del problema

**El problema:** Un sistema necesita leer dígitos escritos a mano (del 0 al 9) para procesarlos automáticamente. Le damos una imagen de un dígito y tiene que decirnos qué número es.

**¿Qué tipo de problema es?**

Clasificación supervisada con **10 clases posibles** (una por cada dígito). Esto lo hace más complejo que un problema binario o de tres clases: el modelo tiene que aprender a distinguir entre diez categorías distintas.

**¿Cómo se representa una imagen en Python?**

Una imagen en escala de grises no es más que una tabla de números. Cada número representa la intensidad de un píxel: 0 es negro, 255 es blanco, y los valores intermedios son grises. El dataset que vamos a usar contiene imágenes de **8×8 píxeles**, es decir, cada imagen es una matriz de 64 valores numéricos.

Para pasarle estos datos a un modelo de ML, esa matriz 8×8 se "aplana" en un vector de 64 valores. Así, cada imagen se convierte en una fila con 64 columnas, igual que cualquier otro dataset tabular.

---
## Fase 2 — Carga y exploración de datos

### 2.1 Importar librerías

Usamos las mismas librerías habituales, con una diferencia: esta vez importamos `RandomForestClassifier` en lugar de KNN. Lo explicaremos en la fase de entrenamiento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("✅ Librerías importadas correctamente")

### 2.2 Cargar el dataset de dígitos

El dataset `digits` de scikit-learn contiene **1.797 imágenes** de dígitos escritos a mano, cada una de 8×8 píxeles. Es una versión reducida del famoso dataset MNIST (que tiene 70.000 imágenes de 28×28), pero perfecta para aprender en clase.

Vamos a ver cómo están organizados los datos antes de convertirlos a DataFrame.

In [ ]:
digits = load_digits()

print("=== Estructura del dataset ===")
print(f"Número de imágenes:      {digits.images.shape[0]}")
print(f"Tamaño de cada imagen:   {digits.images.shape[1]}x{digits.images.shape[2]} píxeles")
print(f"Clases posibles:         {digits.target_names}")
print()
print("=== Datos 'aplanados' para el modelo ===")
print(f"Forma de X (datos aplanados): {digits.data.shape}")
print("Cada fila = una imagen. Cada columna = un píxel.")

### 2.3 Visualizar algunas imágenes del dataset

Esta es una de las grandes ventajas de trabajar con datos de imagen: podemos ver exactamente con qué está trabajando el modelo. Antes de entrenar nada, vamos a mostrar las primeras 20 imágenes del dataset junto con su etiqueta real.

Esto también nos ayuda a desarrollar intuición sobre la dificultad del problema: ¿son fáciles de distinguir a simple vista? ¿Qué pares de dígitos se parecen más entre sí?

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(10, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray_r')  # gray_r: negro=valor alto
    ax.set_title(f'Etiqueta: {digits.target[i]}', fontsize=11)
    ax.axis('off')

plt.suptitle('Primeras 20 imágenes del dataset', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 2.4 ¿Cómo ve el modelo lo que nosotros vemos como imagen?

Para nosotros, el dígito anterior es una imagen con una forma reconocible. Para el modelo, no es más que una lista de 64 números. Vamos a mostrar los valores reales de una imagen concreta para entender este punto clave.

Esta diferencia entre "lo que vemos nosotros" y "lo que procesa el modelo" es fundamental para entender por qué las redes neuronales convolucionales (CNNs) fueron un avance tan grande: aprenden a detectar patrones espaciales en las imágenes, algo que los modelos clásicos no hacen.

In [ ]:
idx = 0  # Cambia este número para ver otras imágenes

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Izquierda: la imagen como la vemos nosotros
axes[0].imshow(digits.images[idx], cmap='gray_r')
axes[0].set_title(f'Lo que vemos nosotros\n(etiqueta real: {digits.target[idx]})', fontsize=11)
axes[0].axis('off')

# Derecha: la matriz de píxeles que ve el modelo
im = axes[1].imshow(digits.images[idx], cmap='Blues')
for i in range(8):
    for j in range(8):
        axes[1].text(j, i, int(digits.images[idx][i, j]),
                     ha='center', va='center', fontsize=9, color='black')
axes[1].set_title('Lo que procesa el modelo\n(matriz de 8×8 valores)', fontsize=11)
axes[1].axis('off')

plt.suptitle('Dos formas de ver lo mismo', fontsize=12)
plt.tight_layout()
plt.show()

### 2.5 Distribución de clases

Antes de continuar, comprobamos si el dataset está balanceado, es decir, si hay aproximadamente el mismo número de ejemplos de cada dígito. Un desbalance fuerte (por ejemplo, 1.000 imágenes del dígito 0 y solo 50 del dígito 9) haría que el modelo aprendiera mal y que la métrica de exactitud nos engañara.

In [ ]:
conteos = pd.Series(digits.target).value_counts().sort_index()

plt.figure(figsize=(8, 3))
conteos.plot(kind='bar', color='steelblue', edgecolor='white')
plt.xlabel('Dígito')
plt.ylabel('Número de imágenes')
plt.title('Distribución de clases en el dataset')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(conteos)

> **Conclusión:** El dataset está bastante equilibrado (entre 174 y 183 imágenes por clase). Esto nos permite usar la exactitud como métrica principal sin que nos engañe.

---
## Fase 3 — Preprocesamiento

### 3.1 Preparar X e y

Los datos ya vienen "aplanados" en `digits.data`: cada imagen es una fila de 64 valores. Esto es exactamente lo que necesita cualquier clasificador clásico de scikit-learn.

Si estuviéramos usando una red neuronal convolucional (CNN), mantendríamos la forma 8×8 para que el modelo pueda detectar patrones espaciales. Pero con los modelos que usamos hoy, trabajamos con el vector plano de 64 características.

In [ ]:
X = digits.data    # (1797, 64) — cada imagen como vector de 64 píxeles
y = digits.target  # (1797,)    — dígito real de cada imagen

print(f"X shape: {X.shape}  → {X.shape[0]} imágenes, {X.shape[1]} píxeles por imagen")
print(f"y shape: {y.shape}  → una etiqueta (0-9) por imagen")
print(f"Rango de valores en X: {X.min()} a {X.max()}")

### 3.2 División en entrenamiento y prueba

Mismo principio que siempre: el modelo aprende con el 80% y evaluamos con el 20% que nunca ha visto. Usamos `stratify=y` para asegurarnos de que todos los dígitos están representados en ambos conjuntos, algo especialmente importante con 10 clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} imágenes")
print(f"Prueba:        {X_test.shape[0]} imágenes")

### 3.3 Normalización

Los valores de píxel van de 0 a 16 en este dataset. Aunque la escala es la misma para todos los píxeles (no hay variables en escalas muy distintas como en Iris), normalizar sigue siendo una buena práctica: acelera el aprendizaje en muchos modelos y mejora la estabilidad numérica.

Aplicamos el mismo criterio de siempre: ajustamos el scaler solo con los datos de entrenamiento.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Rango original en X_train:        {X_train.min():.1f} a {X_train.max():.1f}")
print(f"Rango tras escalar en X_train:    {X_train_scaled.min():.2f} a {X_train_scaled.max():.2f}")

---
## Fase 4 — Entrenamiento del modelo

### 4.1 ¿Por qué Random Forest y no KNN?

Con 64 variables y 1.797 muestras, KNN empieza a tener un problema: calcular distancias en un espacio de 64 dimensiones es costoso y cada vez menos informativo (el fenómeno conocido como *la maldición de la dimensionalidad*).

**Random Forest** es un algoritmo diferente. En lugar de medir distancias, construye muchos árboles de decisión (en este caso, 100) de forma aleatoria y combina sus votos para dar la predicción final. Sus ventajas para este problema son:

- **Robusto en espacios de alta dimensión:** No le afecta tanto tener muchas variables
- **No necesita escalar los datos** (aunque lo hemos hecho igualmente por coherencia)
- **Rápido de entrenar** incluso con cientos de árboles
- **Proporciona importancia de variables:** Puede decirte qué píxeles son más relevantes para clasificar

El parámetro principal es `n_estimators`: cuántos árboles construye el bosque. Más árboles = más estabilidad, pero más tiempo de cómputo.

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100,   # 100 árboles de decisión
    random_state=42     # reproducibilidad
)

modelo.fit(X_train_scaled, y_train)

print("✅ Modelo entrenado correctamente")
print(f"Algoritmo:       {modelo.__class__.__name__}")
print(f"Número de árboles: {modelo.n_estimators}")

### 4.2 Predicciones y comparación visual

Una vez entrenado el modelo, le pasamos las imágenes de test. Para hacer la evaluación más intuitiva, vamos a mostrar algunas de esas imágenes junto con la etiqueta real y la que ha predicho el modelo. Los errores se marcarán en rojo para que sean fáciles de identificar.

In [ ]:
y_pred = modelo.predict(X_test_scaled)

# Mostramos las primeras 20 imágenes de test con sus predicciones
fig, axes = plt.subplots(4, 5, figsize=(11, 9))
axes = axes.flatten()

for i, ax in enumerate(axes):
    # Recuperamos la imagen original (sin escalar) para visualizarla
    imagen = X_test[i].reshape(8, 8)
    ax.imshow(imagen, cmap='gray_r')

    correcto = y_pred[i] == y_test[i]
    color = 'green' if correcto else 'red'
    simbolo = '✓' if correcto else '✗'

    ax.set_title(
        f'Real: {y_test[i]}  Pred: {y_pred[i]} {simbolo}',
        fontsize=9, color=color
    )
    ax.axis('off')

plt.suptitle('Predicciones del modelo sobre imágenes de test', fontsize=13)
plt.tight_layout()
plt.show()

---
## Fase 5 — Evaluación del modelo

### 5.1 Exactitud global

Con 10 clases posibles, un modelo que eligiera al azar acertaría el 10% de las veces. Un modelo útil debería estar muy por encima de eso. Veamos cuánto conseguimos.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud del modelo en test: {accuracy:.2%}")
print(f"(Un modelo aleatorio acertaría aproximadamente el 10%)")

### 5.2 Informe de clasificación por dígito

Con 10 clases, el informe de clasificación es especialmente útil: nos muestra qué dígitos el modelo clasifica bien y cuáles le cuestan más. Esto es información accionable: si el modelo falla mucho con el 4 y el 9, podríamos añadir más ejemplos de esos dígitos al entrenamiento o revisar su calidad.

In [ ]:
print("=== Informe de Clasificación por Dígito ===")
print(classification_report(
    y_test, y_pred,
    target_names=[str(i) for i in range(10)]
))

### 5.3 Matriz de confusión

Con 10 clases la matriz de confusión tiene 10×10 = 100 celdas. El mapa de calor nos permite detectar de un vistazo dónde se concentran los errores.

Lo que buscamos es que los colores más oscuros estén en la diagonal (aciertos) y que fuera de ella todo sea lo más claro posible. Si dos dígitos se confunden entre sí con frecuencia, veremos valores altos en posiciones simétricas fuera de la diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10),
    yticklabels=range(10)
)
plt.xlabel('Dígito Predicho', fontsize=11)
plt.ylabel('Dígito Real', fontsize=11)
plt.title('Matriz de Confusión — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

> **¿Qué pares de dígitos se confunden más?** Fíjate en si el modelo confunde el 3 con el 5, o el 8 con el 9. Estos errores tienen sentido intuitivo: visualmente, algunos dígitos escritos a mano se parecen mucho. La matriz nos dice exactamente cuántas veces ocurre cada tipo de confusión.

### 5.4 ¿Qué píxeles importan más? (Importancia de variables)

Una de las ventajas de Random Forest es que puede decirnos qué variables ha considerado más útiles para clasificar. En este caso, las variables son los 64 píxeles de la imagen.

Si representamos la importancia de cada píxel como un mapa de calor con la forma 8×8 de la imagen original, podemos ver visualmente **qué zonas de la imagen son más informativas** para distinguir los dígitos.

Intuitivamente esperamos que los píxeles del borde exterior sean poco útiles (siempre están en blanco) y que los del centro sean los más importantes.

In [ ]:
importancias = modelo.feature_importances_.reshape(8, 8)

plt.figure(figsize=(5, 4))
plt.imshow(importancias, cmap='hot', interpolation='nearest')
plt.colorbar(label='Importancia relativa')
plt.title('Importancia de cada píxel para la clasificación')
plt.axis('off')
plt.tight_layout()
plt.show()

# Píxel más importante
idx_max = modelo.feature_importances_.argmax()
fila, col = divmod(idx_max, 8)
print(f"Píxel más informativo: posición fila {fila}, columna {col}")

### 5.5 Visualizar los errores del modelo

Quizá lo más valioso de trabajar con imágenes es que podemos *ver* los errores. Cuando el modelo se equivoca, podemos mirar la imagen y muchas veces entender por qué: la escritura es ambigua, el dígito está inclinado, o simplemente se parece a otro.

Esto es algo que no podemos hacer con datasets puramente numéricos, y es una herramienta poderosa para entender el comportamiento real del modelo.

In [ ]:
# Encontramos los índices donde el modelo se equivocó
indices_error = np.where(y_pred != y_test)[0]
print(f"Total de errores: {len(indices_error)} de {len(y_test)} imágenes de test")
print()

# Mostramos hasta 15 errores
n_mostrar = min(15, len(indices_error))
fig, axes = plt.subplots(3, 5, figsize=(11, 7))
axes = axes.flatten()

for i, idx in enumerate(indices_error[:n_mostrar]):
    imagen = X_test[idx].reshape(8, 8)
    axes[i].imshow(imagen, cmap='gray_r')
    axes[i].set_title(
        f'Real: {y_test[idx]}\nPred: {y_pred[idx]}',
        fontsize=10, color='red'
    )
    axes[i].axis('off')

plt.suptitle('Imágenes donde el modelo se equivocó', fontsize=13)
plt.tight_layout()
plt.show()

---
## Fase 6 — Reflexión y próximos pasos

### ¿Qué hemos construido?

Un sistema de reconocimiento de dígitos escritos a mano que funciona con imágenes reales. Hemos aplicado el mismo flujo de trabajo estándar de ML, pero sobre un tipo de dato completamente distinto: imágenes en lugar de medidas numéricas simples.

### Lo nuevo que ha aparecido en este proyecto

- **Datos de imagen:** Cada muestra es una matriz de píxeles, no una lista de medidas. El modelo no "ve" la imagen, trabaja con números.
- **Clasificación con 10 clases:** La evaluación se complica. La matriz de confusión crece a 10×10 y el informe de clasificación tiene una fila por dígito.
- **Importancia de variables:** Random Forest nos dice qué zonas de la imagen son más útiles, algo interpretable e intuitivo.
- **Análisis de errores visual:** Podemos ver exactamente qué imágenes falló el modelo y muchas veces entender por qué.

### ¿Qué vendría después en un proyecto real?

- **Imágenes más grandes y reales:** El dataset MNIST tiene 70.000 imágenes de 28×28. Otros datasets tienen imágenes de cientos de píxeles con color (RGB).
- **Redes neuronales convolucionales (CNNs):** Los modelos clásicos como Random Forest no aprovechan la estructura espacial de la imagen. Las CNNs sí lo hacen, y son el estándar actual para visión por computador.
- **Aumento de datos (Data Augmentation):** Generar variaciones artificiales de las imágenes (rotadas, con ruido, desplazadas) para que el modelo aprenda a ser más robusto.

---

### Pregunta de reflexión final

> Hemos visto que el modelo comete algunos errores. Mira las imágenes donde falló: ¿tú también las habrías leído mal? ¿Qué dice eso sobre la dificultad del problema y sobre los límites de cualquier sistema de reconocimiento automático, incluso los más avanzados?

---
## Ejercicio propuesto (opcional / ampliación)

El modelo actual clasifica cualquier imagen de test de golpe. Vamos a hacer algo más interactivo: **crea una función que reciba un índice de imagen y devuelva la predicción del modelo para esa imagen concreta**, mostrando también la imagen y si el modelo ha acertado o no.

```python
def predecir_digito(indice):
    """
    Muestra la imagen de test en el índice dado
    y la predicción del modelo.
    """
    # Tu código aquí
    pass

# Prueba con varios índices
predecir_digito(0)
predecir_digito(5)
predecir_digito(42)
```

Preguntas adicionales:
1. ¿Qué pasa si usas KNN en lugar de Random Forest con este dataset? ¿Es más lento? ¿Más preciso?
2. ¿Qué ocurre si entrenas el modelo **sin** normalizar los datos? ¿Cambia la exactitud?
3. ¿Puedes encontrar manualmente en el dataset una imagen del dígito 1 que el modelo confunda con otro número?